# Chapter 9: Unconstrained Minimization

## Overview
This chapter marks the beginning of Part III: Algorithms. We focus on solving the unconstrained optimization problem:

$$
\text{minimize} \quad f(x)
$$

where $f: \mathbb{R}^n \to \mathbb{R}$ is convex and twice continuously differentiable. We assume an optimal point $x^\star$ exists. 

### Key Concepts

1. **Optimality Condition:** For an unconstrained problem, the necessary and sufficient condition for $x^\star$ to be optimal is simply that the gradient vanishes:

$$
\nabla f(x^\star) = 0
$$

2. **Descent Methods:** Algorithms that produce a sequence $x^{(k)}$ where $f(x^{(k+1)}) < f(x^{(k)})$. The update rule is generally:

$$
x^{(k+1)} = x^{(k)} + t^{(k)} \Delta x^{(k)}
$$

   where $\Delta x^{(k)}$ is the **step direction** and $t^{(k)} > 0$ is the **step size** (found via exact or backtracking line search).

3. **Gradient Descent:** The simplest descent method where the step direction is the negative gradient:

$$
\Delta x = -\nabla f(x)
$$
   
   - **Pros:** Simple to compute.
   - **Cons:** Can be very slow (zigzagging) if the condition number of the Hessian $\nabla^2 f(x)$ is large (i.e., poorly scaled problems).

4. **Newton's Method:** Uses the second derivative (Hessian) to form a quadratic approximation of $f$ near $x$. The step direction is:

$$
\Delta x_{\text{nt}} = -[\nabla^2 f(x)]^{-1} \nabla f(x)
$$
   
   - **Pros:** Affine invariant (scaling doesn't affect it). Extremely fast convergence near the optimum (quadratic convergence).
   - **Cons:** Requires computing and inverting the Hessian matrix, which is expensive for high-dimensional problems.

## Applications & Problems Solved
- **Foundational Algorithm:** Newton's method is the core engine inside almost all modern interior-point solvers for constrained convex optimization.
- **Machine Learning:** Variants of Gradient Descent (like SGD, Adam) are the workhorses for training deep neural networks.

## Code Example
See `unconstrained_minimization.py` for a visual comparison between **Gradient Descent** and **Newton's Method** on a poorly conditioned quadratic function. You will clearly see the zigzagging behavior of Gradient Descent compared to the direct, one-step convergence of Newton's Method.

![Descent Paths](descent_paths.png)


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def f(x, P, q):
    return 0.5 * x.T @ P @ x + q.T @ x

def grad_f(x, P, q):
    return P @ x + q

def hessian_f(x, P):
    return P

def backtracking_line_search(x, dx, P, q, alpha=0.1, beta=0.7):
    t = 1.0
    val_x = f(x, P, q)
    grad_x = grad_f(x, P, q)
    
    while f(x + t * dx, P, q) > val_x + alpha * t * grad_x.T @ dx:
        t *= beta
    return t

def gradient_descent(x0, P, q, iters=30):
    x = x0.copy()
    path = [x.copy()]
    for _ in range(iters):
        dx = -grad_f(x, P, q)
        t = backtracking_line_search(x, dx, P, q)
        x = x + t * dx
        path.append(x.copy())
    return np.array(path)

def newtons_method(x0, P, q, iters=5):
    x = x0.copy()
    path = [x.copy()]
    for _ in range(iters):
        g = grad_f(x, P, q)
        H = hessian_f(x, P)
        dx = -np.linalg.solve(H, g)
        t = backtracking_line_search(x, dx, P, q)
        x = x + t * dx
        path.append(x.copy())
    return np.array(path)

def demonstrate_algorithms():
    # Define a poorly conditioned quadratic problem
    # P must be symmetric positive definite
    P = np.array([[20.0, 0.0], 
                  [0.0, 1.0]])
    q = np.array([0.0, 0.0])
    
    x0 = np.array([8.0, 8.0])
    
    path_gd = gradient_descent(x0, P, q, iters=40)
    path_nt = newtons_method(x0, P, q, iters=5)
    
    # Plotting
    X1, X2 = np.meshgrid(np.linspace(-10, 10, 100), np.linspace(-10, 10, 100))
    Z = 0.5 * (P[0,0]*X1**2 + P[1,1]*X2**2)
    
    plt.figure(figsize=(10, 8))
    plt.contour(X1, X2, Z, levels=np.logspace(0, 3, 20), cmap='viridis', alpha=0.5)
    
    # Plot paths
    plt.plot(path_gd[:, 0], path_gd[:, 1], 'ro-', markersize=4, label='Gradient Descent')
    plt.plot(path_nt[:, 0], path_nt[:, 1], 'bs-', markersize=6, label="Newton's Method")
    
    # Optimum
    plt.plot(0, 0, 'k*', markersize=15, label='Optimum')
    
    plt.title("Gradient Descent vs Newton's Method")
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.legend()
    plt.grid(True)
    
    plt.savefig("descent_paths.png")
    print("Optimization complete. Plot saved as descent_paths.png")

if __name__ == "__main__":
    demonstrate_algorithms()
